# SEACO-6161 · Generic VACUUM cleanup

Works on **the workspace it runs in**: no workspace config, no `%run`, no audit tables. Import it into any
workspace (OPAS, ARC, ...) and run it on a **Single User** cluster.

1. **Inventory:** every table in the chosen databases with its row count, live size, on-disk size, old-version
   overhead and file count, grouped by database.
2. **Clone check:** finds shallow-clone sources. A VACUUM on a source deletes files its clones still read,
   so these are never VACUUMed.
3. **Candidates:** Delta tables whose old versions take at least `min_overhead_gib`, largest first, at most
   `max_tables`.
4. **VACUUM** with `RETAIN retain_hours HOURS`, printing rows, size and files before and after for each table.
   - `dry_run = true` (the default): runs `VACUUM … DRY RUN`. Nothing is deleted. "Freed" is the estimate
     (the old-version overhead).
   - `dry_run = false`: needs `confirm_workspace` = this workspace's URL (printed in the setup cell). Rows are
     counted by reading the data files, on the same table version, before and after. If they differ the run
     stops.
5. **Summary:** GiB freed in total and per database, and how many databases and tables were freed.

VACUUM never goes below 168 hours, and the notebook stops if `retentionDurationCheck` has been disabled.
Concurrent writes are safe: VACUUM only deletes files that have been unused for longer than the retention.

In [0]:
dbutils.widgets.text("databases", "", "1. Databases (comma list, empty = all)")
dbutils.widgets.text("exclude_tables", "", "2. Never VACUUM (db.table, comma list)")
dbutils.widgets.text("retain_hours", "720", "3. Retain hours (>= 168)")
dbutils.widgets.text("min_overhead_gib", "1", "4. VACUUM only if old versions >= GiB")
dbutils.widgets.text("max_tables", "20", "5. Max tables to VACUUM")
dbutils.widgets.dropdown("clone_check_scope", "all", ["all", "selected"], "6. Clone check: all or selected dbs")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "7. Dry run")
dbutils.widgets.text("confirm_workspace", "", "8. Workspace URL (needed when dry_run = false)")

In [0]:
import datetime as dt
import time
from concurrent.futures import ThreadPoolExecutor

from pyspark.sql import functions as F

GIB = 1024 ** 3
MIN_RETAIN_HOURS = 168
THREADS = 16
META_KEY = "spark.databricks.delta.optimizeMetadataQuery.enabled"

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl", "unknown")
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
RUN_USER = _ctx.userName().get() if _ctx.userName().isDefined() else "unknown"

DATABASES_ARG = [d.strip() for d in dbutils.widgets.get("databases").split(",") if d.strip()]
EXCLUDE = {t.strip().lower() for t in dbutils.widgets.get("exclude_tables").split(",") if t.strip()}
RETAIN_HOURS = int(dbutils.widgets.get("retain_hours"))
MIN_OVERHEAD = float(dbutils.widgets.get("min_overhead_gib")) * GIB
MAX_TABLES = int(dbutils.widgets.get("max_tables"))
CLONE_SCOPE = dbutils.widgets.get("clone_check_scope")
DRY_RUN = dbutils.widgets.get("dry_run") != "false"
CONFIRM = dbutils.widgets.get("confirm_workspace").strip()

_jvm = spark._jvm
_hconf = spark._jsc.hadoopConfiguration()


def human(n):
    if n is None:
        return "-"
    n = float(n)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB", "PiB"):
        if abs(n) < 1024 or unit == "PiB":
            return f"{n:,.2f} {unit}"
        n /= 1024


def num(n):
    return f"{n:,}" if n is not None else "-"


def qname(db, table=None):
    return ".".join(f"`{p}`" for p in [CATALOG, db] + ([table] if table else []))


def list_databases():
    return sorted(r[0] for r in spark.sql(f"SHOW DATABASES IN `{CATALOG}`").collect())


def list_tables(db):
    try:
        rows = spark.sql(f"SHOW TABLES IN {qname(db)}").collect()
    except Exception as e:
        print(f"  ! SHOW TABLES {db}: {str(e)[:200]}")
        return []
    try:
        views = {r["viewName"] for r in spark.sql(f"SHOW VIEWS IN {qname(db)}").collect()}
    except Exception:
        views = set()
    return sorted(r["tableName"] for r in rows if not r["isTemporary"] and r["tableName"] not in views)


def content_summary(path):
    """Recursive size and file count of a path (driver, JVM side)."""
    p = _jvm.org.apache.hadoop.fs.Path(path)
    cs = p.getFileSystem(_hconf).getContentSummary(p)
    return int(cs.getLength()), int(cs.getFileCount())


def normalize(path):
    p = (path or "").strip()
    if p.startswith("dbfs:"):
        p = p[5:]
    return p.rstrip("/")


def is_under(path, parent):
    path, parent = normalize(path), normalize(parent)
    return bool(parent) and (path == parent or path.startswith(parent + "/"))

In [0]:
print(f"Workspace URL : {WORKSPACE_URL}")
print(f"Catalog       : {CATALOG}")
print(f"Run as        : {RUN_USER}")
print(f"Mode          : {'DRY RUN (nothing is deleted)' if DRY_RUN else 'REAL VACUUM'}")
print(f"Retain        : {RETAIN_HOURS} hours ({RETAIN_HOURS / 24:.0f} days of time travel kept)")
print(f"Candidates    : overhead >= {human(MIN_OVERHEAD)}, at most {MAX_TABLES} tables")

In [0]:
problems = []
if RETAIN_HOURS < MIN_RETAIN_HOURS:
    problems.append(f"retain_hours must be >= {MIN_RETAIN_HOURS}")
if spark.conf.get("spark.databricks.delta.retentionDurationCheck.enabled", "true").lower() != "true":
    problems.append("spark.databricks.delta.retentionDurationCheck.enabled has been set to false on this cluster")
if not DRY_RUN and CONFIRM != WORKSPACE_URL:
    problems.append(f"dry_run = false needs confirm_workspace = {WORKSPACE_URL}")
if problems:
    raise Exception("Safety gates failed:\n  - " + "\n  - ".join(problems))

DATABASES = DATABASES_ARG or list_databases()
missing = sorted(set(DATABASES) - set(list_databases()))
if missing:
    raise Exception(f"Databases not found in {CATALOG}: {', '.join(missing)}")

In [0]:
%scala
// Sizing on the EXECUTORS: one Spark task per group of paths, so large folders are sized in parallel.
// Input : temp view _size_input(path)   Output: temp view _size_output(path, bytes, files, error)
// The Hadoop conf travels as plain strings in the task closure; broadcasting a Configuration breaks on
// clusters with spark.serializer = KryoSerializer (Sedona).
import org.apache.hadoop.fs.Path
import org.apache.hadoop.conf.Configuration
import scala.collection.JavaConverters._
import spark.implicits._

def sizePathsToView(inputView: String, outputView: String): Unit = {
  val paths = spark.table(inputView).select("path").as[String].collect().toSeq.distinct
  val confEntries: Map[String, String] =
    spark.sparkContext.hadoopConfiguration.iterator.asScala.map(e => e.getKey -> e.getValue).toMap
  val slices = math.max(1, math.min(paths.length, spark.sparkContext.defaultParallelism * 4))
  val rows = if (paths.isEmpty) Array.empty[(String, Option[Long], Option[Long], String)] else
    spark.sparkContext.parallelize(paths, slices).mapPartitions { it =>
      val hconf = new Configuration()
      confEntries.foreach { case (k, v) => hconf.set(k, v) }
      it.map { p =>
        try {
          val hp = new Path(p)
          val cs = hp.getFileSystem(hconf).getContentSummary(hp)
          (p, Some(cs.getLength), Some(cs.getFileCount), null: String)
        } catch {
          case e: Throwable => (p, None, None, String.valueOf(e.getMessage).take(300))
        }
      }
    }.collect()
  rows.toSeq.toDF("path", "bytes", "files", "error").createOrReplaceTempView(outputView)
  println(s"sized ${paths.length} paths, ${rows.count(_._4 != null)} errors")
}

In [0]:
%scala
// Shallow-clone check. A table whose current files live OUTSIDE its own folder is a shallow clone; the
// folder it points into belongs to the SOURCE table, and a VACUUM on that source would break the clone.
// Input : temp view _clone_check_input(table_name, table_location)
// Output: temp view _clone_check_output(table_name, referenced_dir, n_files, error)
def normalizeDbfs(p: String): String = {
  var s = Option(p).getOrElse("")
  if (s.startsWith("dbfs:")) s = s.substring(5)
  s.stripSuffix("/")
}

def findShallowCloneRefs(): Unit = {
  val input = spark.table("_clone_check_input").as[(String, String)].collect()
  val results = input.par.flatMap { case (name, location) =>
    try {
      val loc = normalizeDbfs(location)
      val outside = spark.read.format("delta").load(location).inputFiles
        .map(normalizeDbfs).filterNot(f => f.startsWith(loc + "/"))
      outside.groupBy(f => f.substring(0, math.max(f.lastIndexOf('/'), 0)))
        .map { case (dir, fs) => (name, dir, fs.length.toLong, null: String) }.toSeq
    } catch {
      case e: Throwable => Seq((name, null: String, 0L, String.valueOf(e.getMessage).take(300)))
    }
  }.seq.toSeq
  results.toDF("table_name", "referenced_dir", "n_files", "error").createOrReplaceTempView("_clone_check_output")
  println(s"clone check: ${input.length} Delta tables checked, ${results.count(_._2 != null)} external file groups, " +
          s"${results.count(_._4 != null)} errors")
}

## 1. Inventory: databases, tables, row counts and sizes

In [0]:
def detail(db, t):
    row = dict(database=db, table=t, format=None, location=None, live_bytes=None, live_files=None,
               last_modified=None, row_count=None, error=None)
    try:
        d = spark.sql(f"DESCRIBE DETAIL {qname(db, t)}").first().asDict()
        row.update(format=(d.get("format") or "").lower() or None, location=d.get("location"),
                   live_bytes=d.get("sizeInBytes"), live_files=d.get("numFiles"),
                   last_modified=d.get("lastModified"))
        row["row_count"] = spark.sql(f"SELECT count(*) AS c FROM {qname(db, t)}").first()["c"]
    except Exception as e:
        row["error"] = str(e)[:300]
    return row


todo = [(db, t) for db in DATABASES for t in list_tables(db)]
t0 = time.time()
with ThreadPoolExecutor(THREADS) as pool:
    INVENTORY = list(pool.map(lambda x: detail(*x), todo))

# On-disk size of every table folder and its _delta_log, sized on the executors by the next cell.
size_paths = set()
for r in INVENTORY:
    if r["location"]:
        size_paths.add(r["location"])
        if r["format"] == "delta":
            size_paths.add(r["location"].rstrip("/") + "/_delta_log")
spark.createDataFrame([(p,) for p in sorted(size_paths)], "path string").createOrReplaceTempView("_size_input")
print(f"{len(INVENTORY):,} tables in {len(DATABASES)} databases described in {time.time() - t0:,.0f}s; "
      f"sizing {len(size_paths):,} folders")

In [0]:
%scala
sizePathsToView("_size_input", "_size_output")

In [0]:
sizes = {r["path"]: r for r in spark.table("_size_output").collect()}
for r in INVENTORY:
    loc = r["location"]
    s = sizes.get(loc) if loc else None
    lg = sizes.get(loc.rstrip("/") + "/_delta_log") if loc and r["format"] == "delta" else None
    r["disk_bytes"] = s["bytes"] if s else None
    r["disk_files"] = s["files"] if s else None
    r["log_bytes"] = lg["bytes"] if lg else None
    if s and s["error"] and not r["error"]:
        r["error"] = f"sizing: {s['error']}"
    if r["format"] == "delta" and r["disk_bytes"] is not None and r["live_bytes"] is not None:
        r["overhead_bytes"] = max(0, r["disk_bytes"] - r["live_bytes"] - (r["log_bytes"] or 0))
    else:
        r["overhead_bytes"] = None

INV_COLS = ["database", "table", "format", "row_count", "live_bytes", "live_files", "disk_bytes", "disk_files",
            "log_bytes", "overhead_bytes", "last_modified", "location", "error"]
inventory_df = spark.createDataFrame(
    [tuple(r[c] for c in INV_COLS) for r in INVENTORY],
    "database string, table string, format string, row_count long, live_bytes long, live_files long, "
    "disk_bytes long, disk_files long, log_bytes long, overhead_bytes long, last_modified timestamp, "
    "location string, error string",
)

In [0]:
print(f"{'table':55s} {'format':7s} {'rows':>15s} {'live':>12s} {'on disk':>12s} {'old versions':>13s} {'files':>11s}")
for db in DATABASES:
    sub = sorted((r for r in INVENTORY if r["database"] == db), key=lambda r: r["table"])
    print(f"\n== {db}: {len(sub)} tables, {num(sum(r['row_count'] or 0 for r in sub))} rows, "
          f"{human(sum(r['live_bytes'] or 0 for r in sub))} live, {human(sum(r['disk_bytes'] or 0 for r in sub))} on disk, "
          f"{human(sum(r['overhead_bytes'] or 0 for r in sub))} old versions")
    for r in sub:
        print(f"  {r['table']:53s} {(r['format'] or '-'):7s} {num(r['row_count']):>15s} {human(r['live_bytes']):>12s} "
              f"{human(r['disk_bytes']):>12s} {human(r['overhead_bytes']):>13s} {num(r['disk_files']):>11s}"
              + (f"  ! {r['error'][:80]}" if r["error"] else ""))
print(f"\nTotal: {len(INVENTORY)} tables, {human(sum(r['disk_bytes'] or 0 for r in INVENTORY))} on disk, "
      f"{human(sum(r['overhead_bytes'] or 0 for r in INVENTORY))} in old versions")

In [0]:
display(inventory_df
        .withColumn("live_gib", F.round(F.col("live_bytes") / GIB, 2))
        .withColumn("disk_gib", F.round(F.col("disk_bytes") / GIB, 2))
        .withColumn("overhead_gib", F.round(F.col("overhead_bytes") / GIB, 2))
        .select("database", "table", "format", "row_count", "live_gib", "disk_gib", "overhead_gib", "disk_files",
                "last_modified", "location", "error")
        .orderBy(F.desc_nulls_last("overhead_gib")))

## 2. Shallow-clone check and VACUUM candidates

In [0]:
clone_input = [(f"{r['database']}.{r['table']}", r["location"]) for r in INVENTORY
               if r["format"] == "delta" and r["location"]]
if CLONE_SCOPE == "all":
    # A clone in another database can point into a table selected here, so check every Delta table.
    def delta_location(db, t):
        try:
            d = spark.sql(f"DESCRIBE DETAIL {qname(db, t)}").first().asDict()
            return (f"{db}.{t}", d["location"]) if (d.get("format") or "").lower() == "delta" else None
        except Exception:
            return None
    others = [(db, t) for db in list_databases() if db not in DATABASES for t in list_tables(db)]
    with ThreadPoolExecutor(THREADS) as pool:
        clone_input += [x for x in pool.map(lambda x: delta_location(*x), others) if x]
spark.createDataFrame(clone_input, "table_name string, table_location string") \
    .createOrReplaceTempView("_clone_check_input")

In [0]:
%scala
findShallowCloneRefs()

In [0]:
clone_rows = spark.table("_clone_check_output").collect()
CLONE_ERRORS = [r for r in clone_rows if r["error"]]
CLONE_SOURCES = {}   # "db.table" of a source -> the clones that read its files
for ref in (r for r in clone_rows if r["referenced_dir"]):
    for r in INVENTORY:
        if r["format"] == "delta" and r["location"] and is_under(ref["referenced_dir"], r["location"]):
            CLONE_SOURCES.setdefault(f"{r['database']}.{r['table']}", set()).add(ref["table_name"])

CANDIDATES = sorted(
    [r for r in INVENTORY
     if r["format"] == "delta" and not r["error"]
     and (r["overhead_bytes"] or 0) >= MIN_OVERHEAD
     and f"{r['database']}.{r['table']}".lower() not in EXCLUDE
     and f"{r['database']}.{r['table']}" not in CLONE_SOURCES],
    key=lambda r: -(r["overhead_bytes"] or 0),
)[:MAX_TABLES]

In [0]:
for src, clones in sorted(CLONE_SOURCES.items()):
    print(f"  SKIP clone source {src} (read by {', '.join(sorted(clones))})")
if CLONE_ERRORS:
    print(f"  ⚠️  {len(CLONE_ERRORS)} Delta tables could not be checked for clones, e.g. "
          f"{CLONE_ERRORS[0]['table_name']}: {CLONE_ERRORS[0]['error'][:120]}")
print(f"\n{len(CANDIDATES)} tables to VACUUM ({'dry run' if DRY_RUN else 'REAL'}), "
      f"{human(sum(r['overhead_bytes'] for r in CANDIDATES))} in old versions:")
for r in CANDIDATES:
    print(f"  {r['database'] + '.' + r['table']:70s} {human(r['disk_bytes']):>12s} on disk, "
          f"{human(r['overhead_bytes']):>12s} old versions, {num(r['disk_files'])} files")

## 3. VACUUM, with row counts before and after

In [0]:
class RowCheckFailed(Exception):
    pass


def current_version(q):
    return spark.sql(f"DESCRIBE HISTORY {q} LIMIT 1").first()["version"]


def scan_count(q, version):
    """Row count of one table version, read from the data files (not the Delta statistics)."""
    old = spark.conf.get(META_KEY, "true")
    spark.conf.set(META_KEY, "false")
    try:
        return spark.sql(f"SELECT count(*) AS c FROM {q} VERSION AS OF {version}").first()["c"]
    finally:
        spark.conf.set(META_KEY, old)


spark.conf.set("spark.databricks.delta.vacuum.parallelDelete.enabled", "true")
RESULTS = []
for r in CANDIDATES:
    name, q = f"{r['database']}.{r['table']}", qname(r["database"], r["table"])
    res = dict(database=r["database"], table=r["table"], version=None, rows_before=None, rows_after=None,
               bytes_before=r["disk_bytes"], bytes_after=None, files_before=r["disk_files"], files_after=None,
               freed_bytes=None, files_listed=None, status=None, seconds=None, detail=None)
    t0 = time.time()
    try:
        res["version"] = v = current_version(q)
        if DRY_RUN:
            res["rows_before"] = res["rows_after"] = r["row_count"]   # from Delta statistics
            print(f"\n{name} @ version {v}\n  rows before   : {num(res['rows_before'])}")
            res["files_listed"] = spark.sql(f"VACUUM {q} RETAIN {RETAIN_HOURS} HOURS DRY RUN").count()
            res["freed_bytes"] = r["overhead_bytes"]
            res["bytes_after"] = r["disk_bytes"] - r["overhead_bytes"]
            res["status"] = "dry_run"
            print(f"  DRY RUN       : {num(res['files_listed'])} files listed (max 1,000), "
                  f"about {human(res['freed_bytes'])} would be freed")
            print(f"  rows after    : {num(res['rows_after'])} (unchanged, nothing deleted)")
        else:
            res["rows_before"] = scan_count(q, v)
            print(f"\n{name} @ version {v}\n  rows before   : {num(res['rows_before'])}")
            res["bytes_before"], res["files_before"] = content_summary(r["location"])
            spark.sql(f"VACUUM {q} RETAIN {RETAIN_HOURS} HOURS")
            res["bytes_after"], res["files_after"] = content_summary(r["location"])
            res["freed_bytes"] = res["bytes_before"] - res["bytes_after"]
            res["rows_after"] = scan_count(q, v)
            print(f"  rows after    : {num(res['rows_after'])}")
            print(f"  size          : {human(res['bytes_before'])} -> {human(res['bytes_after'])}, "
                  f"files {num(res['files_before'])} -> {num(res['files_after'])}, freed {human(res['freed_bytes'])}")
            if res["rows_after"] != res["rows_before"]:
                res["status"] = "row_check_failed"
                raise RowCheckFailed(f"DATA CHECK FAILED on {name} @ version {v}: "
                                     f"{res['rows_before']:,} rows before, {res['rows_after']:,} after")
            res["status"] = "ok"
    except RowCheckFailed as e:
        res["detail"] = str(e)
        res["seconds"] = time.time() - t0
        RESULTS.append(res)
        raise
    except Exception as e:
        res["status"], res["detail"] = "error", str(e)[:300]
        print(f"  ! {name}: {str(e)[:300]}")
    res["seconds"] = time.time() - t0
    RESULTS.append(res)

RES_COLS = ["database", "table", "version", "status", "rows_before", "rows_after", "bytes_before", "bytes_after",
            "freed_bytes", "files_before", "files_after", "files_listed", "seconds", "detail"]
results_df = spark.createDataFrame(
    [tuple(x[c] for c in RES_COLS) for x in RESULTS],
    "database string, table string, version long, status string, rows_before long, rows_after long, "
    "bytes_before long, bytes_after long, freed_bytes long, files_before long, files_after long, "
    "files_listed long, seconds double, detail string",
)

In [0]:
print(f"{'table':70s} {'status':17s} {'rows before':>15s} {'rows after':>15s} {'disk before':>12s} "
      f"{'disk after':>12s} {'freed':>12s}")
for x in RESULTS:
    print(f"{x['database'] + '.' + x['table']:70s} {x['status']:17s} {num(x['rows_before']):>15s} "
          f"{num(x['rows_after']):>15s} {human(x['bytes_before']):>12s} {human(x['bytes_after']):>12s} "
          f"{human(x['freed_bytes']):>12s}")

## 4. Summary

In [0]:
done = [x for x in RESULTS if x["status"] in ("ok", "dry_run")]
freed = [x for x in done if (x["freed_bytes"] or 0) > 0]
by_db = {}
for x in freed:
    d = by_db.setdefault(x["database"], dict(tables=0, freed=0))
    d["tables"] += 1
    d["freed"] += x["freed_bytes"]
total = sum(x["freed_bytes"] for x in freed)
word = "Would free (estimate)" if DRY_RUN else "Freed"

print(f"==== Summary ({'DRY RUN, nothing deleted' if DRY_RUN else 'REAL VACUUM'}) on {WORKSPACE_URL} ====")
print(f"Tables VACUUMed   : {len(done)} of {len(CANDIDATES)} candidates "
      f"({sum(x['status'] == 'error' for x in RESULTS)} errors)")
print(f"Row check         : {'not run (dry run)' if DRY_RUN else ('all tables unchanged' if all(x['rows_before'] == x['rows_after'] for x in done) else 'MISMATCH')}")
print(f"{word:18s}: {total / GIB:,.2f} GiB ({human(total)})")
print(f"Databases freed   : {len(by_db)}")
print(f"Tables freed      : {len(freed)}")
for db, d in sorted(by_db.items(), key=lambda kv: -kv[1]["freed"]):
    print(f"  {db:50s} {d['tables']:>4} tables  {d['freed'] / GIB:>12,.2f} GiB")
if DRY_RUN and CANDIDATES:
    print(f"\nTo run for real: dry_run = false, confirm_workspace = {WORKSPACE_URL}")

In [0]:
display(results_df
        .withColumn("disk_before_gib", F.round(F.col("bytes_before") / GIB, 2))
        .withColumn("disk_after_gib", F.round(F.col("bytes_after") / GIB, 2))
        .withColumn("freed_gib", F.round(F.col("freed_bytes") / GIB, 2))
        .select("database", "table", "status", "version", "rows_before", "rows_after", "disk_before_gib",
                "disk_after_gib", "freed_gib", "files_before", "files_after", "files_listed", "seconds", "detail"))